In [1]:

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [2]:

from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

class AgentState(TypedDict):
    user_input: str
    route: str
    output: str
    messages: Annotated[list[AnyMessage], add_messages]


In [3]:

from langchain_core.messages import HumanMessage, SystemMessage

VALID_ROUTES = {
    "seo_blog_writer",
    "x_blog_writer",
    "general"
}

In [4]:
def router_node(state: AgentState):
    router_prompt = """
    You are a content routing classifier.

    Classify the user's latest request into exactly one:

    seo_blog_writer:
      Long-form articles, SEO blogs, content marketing.

    x_blog_writer:
      Tweets, X posts, threads, hashtags, short social posts.

    general:
      Greetings, questions, ordinary conversation,
      memory questions, and other requests.

    Return ONLY one of these exact route names.
    Do not explain your answer.
    """

    response = llm.invoke([
        SystemMessage(content=router_prompt),
        HumanMessage(content=state["user_input"])
    ])

    route = response.content.strip().lower()

    if route not in VALID_ROUTES:
        route = "general"

    return {
        "route": route,
        "messages": [
            HumanMessage(content=state["user_input"])
        ]
    }

In [ ]:
from langchain_core.tools import tool

@tool
def research_tool(topic: str) -> str:
    """Research a topic in depth for a long-form article.
    Use for background facts, themes, and references.
    This is an expensive tool: use at most twice per request.
    """
    return f"""
    Sample research for: {topic}

    Core themes:
    - Definition and background
    - Real-world applications
    - Benefits and limitations
    - Implementation considerations
    - Future directions

    These are illustrative research notes, not
    verified citations or current factual sources.
    """


@tool
def internet_search_tool(query: str) -> str:
    """Perform a lightweight search for SEO keywords,
    related topics, trends, hashtags, or quick facts.
    """
    return f"""
    Sample search for: {query}

    Suggested keyword phrases:
    - AI agents
    - LangGraph workflow
    - Multi-agent architecture
    - Agentic AI applications

    Suggested hashtags:
    #AI #LangGraph #AgenticAI #MachineLearning

    These are examples, not live search results.
    """